<h1 align="center">
  <span style="color:#e91e63">⚔️</span>
  <span style="color:#9c27b0">S</span><span style="color:#673ab7">I</span><span style="color:#3f51b5">G</span><span style="color:#2196f3">I</span><span style="color:#009688">L</span>
  <span style="color:#ff9800">S</span><span style="color:#ff5722">T</span><span style="color:#f44336">R</span><span style="color:#e91e63">I</span><span style="color:#9c27b0">K</span><span style="color:#673ab7">E</span>
  <span style="color:#e91e63">⚔️</span>
  <br/>
  <sub><sup>🧙‍♂️ Colab Notebook 🧙‍♀️</sup></sub>
</h1>

Run all three model tasks (**🎥 collect → 🧠 train → 🔮 inference**) for either the **CNN** or **landmark** pipeline on a Google Colab GPU. The trained weights are auto-downloaded to your local Downloads folder at the end, drop-in compatible with `Teams/Team<N>/models/`.

## 🚀 How to use

1. 🖥️ **Runtime → Change runtime type → GPU** (only needed for CNN training).
2. ⚙️ Run **Section 0** to install dependencies and pick your team / model.
3. 🎚️ Run **Section 1** to set hyperparameters (or just leave the defaults).
4. 📦 Run **Section 2** to load training data — pick **Option A** (upload zip) *or* **Option B** (webcam in browser).
5. 🏋️ Run the **Section 3** cell that matches your `MODEL_TYPE` (CNN ↔ 3A, landmark ↔ 3B). The other one will safely skip.
6. 🔎 (Optional) Run **Section 4** to sanity-check predictions.
7. 💾 Run **Section 5** to download the weights to your PC's Downloads folder.

Cells are split per model: CNN cells and landmark cells live side-by-side and each one self-skips if it doesn't match your selected `MODEL_TYPE`, so **▶️ Run All** does the right thing automatically.

## 📚 Key concepts (read this first if you're new to AI)

This notebook uses standard machine-learning terms. If any aren't familiar, skim this table — every later section assumes them.

| Term | Plain-English meaning |
|---|---|
| 🤖 **Model** | A function we *train* to turn inputs (images or landmarks) into outputs (class labels like `move1`…`move5`). |
| 🎛️ **Weights / parameters** | The numbers *inside* the model that get adjusted during training. The `.pth` / `.pkl` file you download at the end is basically a big bag of these numbers. |
| 🏋️ **Training** | Showing the model lots of labelled examples and nudging its weights so it makes fewer mistakes. |
| 🔁 **Epoch** | One full pass through your training data. More epochs = more learning, but also more risk of *overfitting*. |
| 📦 **Batch** | A small group of training examples processed together. `BATCH_SIZE = 16` means 16 images per step. |
| 📉 **Loss** | A single number measuring how wrong the model is right now. Training drives loss *down*. |
| 🧭 **Optimizer** | The algorithm that updates the weights to reduce loss. **Adam** is the safe default — it auto-tunes per-weight step sizes. |
| 👣 **Learning rate (lr)** | How big a step the optimizer takes. Too high → unstable; too low → painfully slow. Common values: `1e-3` (fast), `1e-5` (cautious). |
| ⬇️ **Gradient** | The direction in weight-space that would *increase* the loss. The optimizer steps in the opposite direction. |
| 🧠💭 **Overfitting** | The model memorizes its training data and flunks new data. Tell-tale sign: train accuracy ≫ validation accuracy. |
| ✅ **Validation set** | A slice of data held back from training and used as an honest scoreboard. |
| 🎚️ **Hyperparameter** | A knob *you* choose before training (epochs, learning rate, etc.) — as opposed to weights, which the optimizer chooses. |
| 🖼️ **CNN** | *Convolutional neural network* — a model specialized for images. MobileNetV2 is one example. |
| 🌍 **ImageNet** | A famous dataset of 1.2 million labelled photos. Models pretrained on it have learned generally-useful visual features. |
| 🔄 **Transfer learning** | Starting from a model already trained on a huge dataset (ImageNet here) and adapting it to your task. Much faster than starting from random weights. |
| 🎲 **Dropout** | A regularization trick that randomly *zeros* some neurons during training, forcing the network not to depend on any single one. |
| 〰️ **ReLU** | A simple non-linear function: `max(0, x)`. Stacked between linear layers, it lets the network learn curved decision boundaries. |
| 📊 **Softmax** | Turns raw model scores into probabilities that sum to 1.0 (used at the very end for multi-class problems). |
| ✋ **Landmark / keypoint** | An (x, y, z) point on the hand (wrist, knuckles, fingertips, …) — produced by MediaPipe. |
| 🌲🌳🌴 **RandomForest** | A "classical" (non-deep) classifier: a committee of decision trees that vote on the answer. |
| 📐 **MediaPipe** | Google's library for real-time perception tasks. We use its hand-landmark detector. |
| 🧮 **Tensor** | A multi-dimensional array — PyTorch's basic data type. Shape `(16, 3, 224, 224)` = 16 colour images of 224 × 224 pixels. |

You don't need to memorize all of this — just know it's here when something looks unfamiliar. 💡

---
## ⚙️ Section 0 · Setup
Installs deps, picks your team + model type, prepares the workspace.

In [ ]:
# --- 0.1  Install dependencies --------------------------------------------
# Torch + torchvision are pre-installed on Colab.  We add the rest:
#   mediapipe         -> hand-landmark detector (landmark pipeline)
#   scikit-learn      -> RandomForest / MLP classifiers (landmark pipeline)
#   joblib            -> serializes sklearn models to .pkl
#   opencv-headless   -> image / webcam frame ops, no GUI dependency
#   pillow            -> required by torchvision.datasets.ImageFolder
%pip install -q mediapipe scikit-learn joblib opencv-python-headless pillow
print('\nDependencies installed.')

In [ ]:
# --- 0.2  >>> EDIT ME <<<  master switches --------------------------------
TEAM_NUM   = 1            # 1..6 -- matches your Teams/TeamN folder on disk
MODEL_TYPE = 'cnn'        # 'cnn'  or  'landmark'

assert MODEL_TYPE in ('cnn', 'landmark'), "MODEL_TYPE must be 'cnn' or 'landmark'"
assert 1 <= TEAM_NUM <= 6, 'TEAM_NUM must be 1..6'
print(f"Configured: Team {TEAM_NUM} | model = {MODEL_TYPE}")

In [ ]:
# --- 0.3  Workspace + GPU check -------------------------------------------
import os, pathlib, sys, shutil, json

WORK_DIR        = pathlib.Path('/content/sigil_strike').resolve()
TEAM_DIR        = WORK_DIR / 'teams' / f'Team{TEAM_NUM}'
IMAGES_DIR      = TEAM_DIR / 'images'                      # CNN data
LANDMARK_CSV    = TEAM_DIR / 'hand_sign_data.csv'          # landmark data
MODELS_OUT_DIR  = WORK_DIR / 'Teams' / f'Team{TEAM_NUM}' / 'models'
MODELS_OUT_DIR.mkdir(parents=True, exist_ok=True)
TEAM_DIR.mkdir(parents=True, exist_ok=True)

import torch
if torch.cuda.is_available():
    DEVICE = torch.device('cuda:0')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print('No GPU detected - CNN training will be SLOW.  Runtime -> Change runtime type -> GPU')

print(f"Workspace : {WORK_DIR}")
print(f"Team dir  : {TEAM_DIR}")
print(f"Models dir: {MODELS_OUT_DIR}")

---
## 🎚️ Section 1 · Hyperparameters

A **hyperparameter** is a setting *you* choose before training — like "how many epochs to run" or "how big a learning step to take". They're different from the model's **weights**, which the optimizer adjusts automatically while training. Picking good hyperparameters often matters more than picking a fancier model. 🎯

Defaults below work well out of the box. Tweak when:

* 📉 **Underfitting** (validation accuracy stays low even on training data): increase epochs, add fine-tune epochs (CNN), increase MLP hidden size or RF `n_estimators` (landmark).
* 📈 **Overfitting** (training accuracy ≫ validation accuracy — the model memorized the training set): reduce epochs, lower learning rate, increase dropout, decrease RF `max_depth`.
* 🐢 **Training is too slow**: increase `CNN_BATCH_SIZE` (if GPU memory allows), reduce `CNN_EPOCHS`.

Each pipeline gets its own cell so you can edit only what's relevant to you. 🧩

### 🅰️ 1A · CNN hyperparameters


In [ ]:
# --- 1A  CNN hyperparameters ---------------------------------------------
# These control the MobileNetV2 transfer-learning run in Section 3A.

CNN_IMG_SIZE        = (224, 224)   # MobileNetV2's native input — don't change unless you also retrain from scratch.
CNN_BATCH_SIZE      = 16           # Larger = faster epochs but more GPU RAM.  Try 32 on a T4 / L4.

# Two-phase training (see Section 3A markdown for what each phase does):
CNN_EPOCHS          = 15           # Phase 1: train the new classifier head only.
CNN_FINETUNE_EPOCHS = 5            # Phase 2: also unfreeze the top of MobileNetV2 and fine-tune.

# Learning rates: phase-1 trains a fresh head (high LR is fine).
# Phase-2 nudges pretrained weights — must be 100×–1000× smaller or you'll wreck them.
CNN_LR_HEAD         = 1e-3
CNN_LR_FINETUNE     = 1e-5

# Regularization in the classifier head.  Bump up if val acc lags train acc:
CNN_DROPOUT_1       = 0.2          # after MobileNet feature pooling
CNN_DROPOUT_2       = 0.3          # before final logits
CNN_HIDDEN_DIM      = 128          # bottleneck between 1280-d features and the class logits

# How many of MobileNetV2's 19 inverted-residual blocks to unfreeze in phase 2.
# 4 is conservative; raise to 6-8 if you have lots of data, drop to 2 if you have very little.
CNN_UNFREEZE_LAST_N = 4

# Augmentation strength — how aggressively we perturb training images.
# Keep modest: hand-shape gestures are sensitive to large rotations / crops.
CNN_AUG_ROT_DEG     = 12
CNN_AUG_TRANSLATE   = (0.08, 0.08)
CNN_AUG_BRIGHTNESS  = 0.15

CNN_VAL_FRACTION    = 0.20         # fraction of dataset held out for validation
CNN_RANDOM_SEED     = 42           # reproducible split + augmentation order

print('CNN hyperparameters set.')

### 🅱️ 1B · Landmark hyperparameters


In [ ]:
# --- 1B  Landmark hyperparameters ----------------------------------------
# These control the sklearn classifier trained in Section 3B.

LANDMARK_MODEL    = 'rf'           # 'rf' = RandomForest (default; robust)
                                    # 'mlp' = small neural net (slightly better with lots of data)

# RandomForest knobs (only used when LANDMARK_MODEL='rf'):
RF_N_ESTIMATORS   = 200            # more trees = smoother decision boundary, slower predict
RF_MAX_DEPTH      = 20             # cap individual tree depth to avoid memorising noise

# MLP knobs (only used when LANDMARK_MODEL='mlp'):
MLP_HIDDEN        = (128, 64)      # widths of hidden layers — bigger fits more complex boundaries
MLP_MAX_ITER      = 500            # cap on optimizer iterations
MLP_EARLY_STOP    = True           # halt when val loss stops improving (avoids overfit)

LANDMARK_TEST_SIZE = 0.20          # held-out fraction for evaluation
LANDMARK_SEED      = 42            # reproducible split

print('Landmark hyperparameters set.')

---
## 📦 Section 2 · Data Collection

Pick **one** of the two options below — you only need to populate the data once.

* 🅰️ **Option A — Upload zip** (recommended): collect data locally with the existing `collect_data.py`, then upload the resulting `TeamN/` folder as a zip.
* 🅱️ **Option B — Webcam in browser**: capture frames live via the Colab webcam bridge. Slower but doesn't require local collection.

> 💡 **How many samples per move?** Aim for **150–300 per class**, collected as **3–5 separate bursts of ~60 frames** rather than one big burst.
>
> - Frames inside a single burst are highly correlated (same lighting, same hand pose, ~1 second apart), so 60 in-burst frames is closer to ~10 truly independent samples. Between bursts, **vary something on purpose**: camera angle, distance, lighting, hand orientation, or who's posing — that variation is what generalizes to real gameplay.
> - Keep classes **balanced** (similar counts per move). If one class dominates, the model leans toward it.
> - Rough tiers:
>   - **~60 / move (1 burst)** — demo only; looks great in Section 4 but breaks under new conditions.
>   - **~150 / move (2–3 bursts)** — workable for a single player, consistent lighting.
>   - **~300 / move (4–5 bursts, varied conditions)** — robust across players and rooms.
>   - **> 500 / move** — diminishing returns with MobileNetV2; spend the effort on *variety* instead.
> - Practical recipe: leave `N_FRAMES = 60` in **2B.4**, run it for each move, then change something (move the camera, swap the player, flip a lamp on) and run another round until each class hits your target.


### 🅰️ Option A · Upload an existing dataset zip

Zip the folder so the structure inside the archive looks like one of these (matches what `collect_data.py` writes):

**🖼️ CNN:**
```
Team1/images/move1/frame_00000.jpg
Team1/images/move1/frame_00001.jpg
Team1/images/move2/...
```

**✋ Landmark:**
```
Team1/hand_sign_data.csv
```

Then run the cell — a file picker will appear. 📂

In [ ]:
# --- 2A.  Upload + extract ------------------------------------------------
from google.colab import files
import zipfile, io

print('Pick your dataset .zip ...')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file was uploaded.')

name, blob = next(iter(uploaded.items()))
print(f'\nExtracting {name} ({len(blob)/1024:.1f} KB) ...')

extract_root = WORK_DIR / 'teams'

# Wipe any previously-extracted data for THIS team so a re-upload starts clean
# (other teams' folders under extract_root are left alone).
if TEAM_DIR.exists():
    print(f'Removing old data at {TEAM_DIR} ...')
    shutil.rmtree(TEAM_DIR)

with zipfile.ZipFile(io.BytesIO(blob)) as zf:
    zf.extractall(extract_root)

# Try to locate the team folder we expect.  If the user zipped *contents*
# of TeamN (vs the TeamN folder itself), fix that up.
if not TEAM_DIR.exists():
    matches = list(extract_root.rglob(f'Team{TEAM_NUM}'))
    if matches:
        shutil.move(str(matches[0]), str(TEAM_DIR))
    else:
        for child in extract_root.iterdir():
            if child.is_dir():
                shutil.move(str(child), str(TEAM_DIR))
                break

if MODEL_TYPE == 'cnn':
    if not IMAGES_DIR.exists():
        raise FileNotFoundError(f'Expected images at {IMAGES_DIR}')
    counts = {p.name: len(list(p.glob('*.jpg'))) for p in sorted(IMAGES_DIR.iterdir()) if p.is_dir()}
    print(f'\nCNN dataset ready: {counts}')
else:
    if not LANDMARK_CSV.exists():
        raise FileNotFoundError(f'Expected CSV at {LANDMARK_CSV}')
    import csv
    from collections import Counter
    with open(LANDMARK_CSV) as f:
        labels = [row[0] for row in csv.reader(f) if row]
    print(f'\nLandmark dataset ready: {dict(Counter(labels))}')


### 🅱️ Option B · Capture frames in the browser (webcam bridge) 📷

Colab can't open `cv2.VideoCapture` directly — there's no kernel-side camera. Instead we ship a JavaScript snippet to the browser that grabs frames from `navigator.mediaDevices.getUserMedia()` and shuttles them back as base64 JPEGs. Run **2B.1** once to set up the bridge, then run the saver that matches your `MODEL_TYPE` (**2B.2** for CNN, **2B.3** for landmark — the other one self-skips), then run **2B.4** for *each* class you want to record. 🎬

#### 📡 2B.1 · Webcam bridge (shared by CNN + landmark)


In [ ]:
# --- 2B.1  Webcam bridge (shared by CNN + landmark) ----------------------
# JavaScript runs in the browser tab; eval_js() lets us call into it from Python.
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np, cv2, time

_WEBCAM_JS = Javascript('''
  async function startStream() {
    if (!window._sigilStream) {
      const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
      const video  = document.createElement('video');
      video.style.display = 'block';
      video.srcObject = stream;
      document.body.appendChild(video);
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      window._sigilStream = stream;
      window._sigilVideo  = video;
    }
    return 'ready';
  }
  async function grabFrame(quality) {
    const v = window._sigilVideo;
    const c = document.createElement('canvas');
    c.width  = v.videoWidth;
    c.height = v.videoHeight;
    c.getContext('2d').drawImage(v, 0, 0);
    return c.toDataURL('image/jpeg', quality);
  }
  async function stopStream() {
    if (window._sigilStream) {
      window._sigilStream.getVideoTracks().forEach(t => t.stop());
      window._sigilVideo.remove();
      window._sigilStream = null;
      window._sigilVideo  = null;
    }
    return 'stopped';
  }
''')

def webcam_start():
    display(_WEBCAM_JS)
    eval_js('startStream()')

def webcam_stop():
    eval_js('stopStream()')

def webcam_grab_bgr(quality: float = 0.85) -> np.ndarray:
    data = eval_js(f'grabFrame({quality})')
    jpg  = b64decode(data.split(',', 1)[1])
    arr  = np.frombuffer(jpg, dtype=np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)   # BGR (OpenCV convention)

print('Webcam bridge ready.  Run 2B.2 (CNN) or 2B.3 (landmark) next.')

#### 🖼️ 2B.2 · CNN saver

For the CNN pipeline, each captured frame is **resized to 224×224** (MobileNetV2's native input) and saved as a JPEG into `teams/Team<N>/images/<class>/`. No landmark extraction at this stage — the network learns the gesture directly from raw pixels, so we just want clean, consistent crops. ✂️

In [ ]:
# --- 2B.2  CNN saver ------------------------------------------------------
if MODEL_TYPE != 'cnn':
    print(f'Skipping (MODEL_TYPE={MODEL_TYPE}).  Run 2B.3 instead.')
else:
    SAVE_SIZE = (224, 224)
    def save_sample(frame_bgr, cls):
        """Resize the BGR frame to 224x224 and write it into the class folder.
        Returns True on success (always succeeds for CNN — no detection required).
        """
        d = IMAGES_DIR / cls; d.mkdir(parents=True, exist_ok=True)
        idx = len(list(d.glob('*.jpg')))
        cv2.imwrite(str(d / f'frame_{idx:05d}.jpg'), cv2.resize(frame_bgr, SAVE_SIZE))
        return True
    print('CNN save_sample() ready.')

#### ✋ 2B.3 · Landmark saver (with MediaPipe)

For the landmark pipeline, raw images are *thrown away* — we only keep a 126-dim feature vector per frame.

1. 📐 **MediaPipe HandLandmarker** locates up to 4 hands and returns 21 (x, y, z) keypoints per hand.
2. 🖐️🤚 We pick the **two largest** hands (by bounding-box area) and order them **left-to-right** (by the wrist's x-coordinate). This gives a stable left/right slot regardless of how MediaPipe internally orders hands.
3. 🎯 Each hand is **normalized**: we subtract the wrist (point 0) so coordinates are *translation-invariant*, then divide by the wrist→middle-finger-MCP distance (point 9) so they're *scale-invariant* — same gesture from a different distance / different person → similar features.
4. 🔗 The two normalized hands are concatenated → 21 × 3 × 2 = **126 features per sample**, appended to `hand_sign_data.csv`.

Frames where two hands aren't visible are silently skipped. 🚫

In [ ]:
# --- 2B.3  Landmark saver -------------------------------------------------
if MODEL_TYPE != 'landmark':
    print(f'Skipping (MODEL_TYPE={MODEL_TYPE}).  Run 2B.2 instead.')
else:
    import urllib.request, csv
    LANDMARKER_PATH = WORK_DIR / 'hand_landmarker.task'
    if not LANDMARKER_PATH.exists():
        url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
        print('Downloading hand_landmarker.task ...')
        urllib.request.urlretrieve(url, LANDMARKER_PATH)

    import mediapipe as mp
    from mediapipe import tasks as mp_tasks

    # num_hands=4 lets the detector pick up extra hands (e.g. an onlooker);
    # we'll filter to the two biggest below.  Confidence thresholds are tuned
    # to be permissive — bad samples are easier to deal with than missed ones.
    _opts = mp_tasks.vision.HandLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=str(LANDMARKER_PATH)),
        num_hands=4, min_hand_detection_confidence=0.7, min_tracking_confidence=0.6)
    DETECTOR = mp_tasks.vision.HandLandmarker.create_from_options(_opts)

    def _bbox_area(lms):
        xs = [lm.x for lm in lms]; ys = [lm.y for lm in lms]
        return (max(xs) - min(xs)) * (max(ys) - min(ys))

    def _two_largest_hands(res):
        # Picks the two most prominent hands and orders them left->right.
        if not res.hand_landmarks or len(res.hand_landmarks) < 2:
            return None
        top = sorted(res.hand_landmarks, key=_bbox_area, reverse=True)[:2]
        top.sort(key=lambda l: l[0].x)
        return top

    def _norm_hand(lms):
        # Translation- and scale-invariant 63-vector for one hand.
        a = np.array([[lm.x, lm.y, lm.z] for lm in lms])
        c = a - a[0]                       # subtract wrist  -> translation-invariant
        s = np.linalg.norm(c[9])           # wrist->middle-MCP distance
        if s > 1e-6: c /= s                # divide by scale -> scale-invariant
        return c.flatten()

    def save_sample(frame_bgr, cls):
        """Detect, normalize, and append a landmark row.  Returns False if
        two hands weren't visible (frame is silently skipped)."""
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        hands = _two_largest_hands(DETECTOR.detect(img))
        if hands is None:
            return False
        feats = np.concatenate([_norm_hand(hands[0]), _norm_hand(hands[1])])
        with open(LANDMARK_CSV, 'a', newline='') as f:
            csv.writer(f).writerow([cls] + feats.tolist())
        return True
    print('Landmark save_sample() ready.')

In [ ]:
# --- 2B.4  Capture a burst of frames for ONE class ------------------------
# Edit CLASS and N_FRAMES, then run.  Re-run with a different CLASS for each move.
CLASS    = 'move1'    # one of: move1 move2 move3 move4 move5
N_FRAMES = 60         # how many frames to capture in this burst
INTERVAL = 0.15       # seconds between captures

assert CLASS in ('move1', 'move2', 'move3', 'move4', 'move5')

webcam_start()
print('Hold the gesture in front of the camera.  Capturing in 2 s ...')
time.sleep(2)
saved = skipped = 0
for i in range(N_FRAMES):
    frame = webcam_grab_bgr()
    if save_sample(frame, CLASS):
        saved += 1
    else:
        skipped += 1
    print(f'  [{i+1:>3}/{N_FRAMES}] saved={saved}  skipped={skipped}', end='\r')
    time.sleep(INTERVAL)
webcam_stop()
print(f'\nDone. {CLASS}: +{saved} samples ({skipped} skipped).')

### 🏋️ 3A · CNN training (MobileNetV2 transfer learning) 🖼️

#### 💬 What's happening in plain English

We're going to take **MobileNetV2** 📱 — an image classifier Google trained on 1.2 million ImageNet photos, so it already "sees" edges, textures, and common objects well — bolt a fresh 5-way classifier on top, and gently teach it to recognize *your* hand gestures. This trick is called **transfer learning** 🔄, and it's how we can get good results from a few hundred training images instead of the millions a CNN trained from scratch would need.

Training happens in **two phases**:

1. 🥇 **Phase 1 — train the head only.** The new 5-way classifier ("the head") starts out as random numbers. We *freeze* 🧊 MobileNetV2 underneath so only the head learns at first. Fast and safe — it can't accidentally wreck MobileNetV2's pretrained knowledge.
2. 🥈 **Phase 2 — gently fine-tune the top of MobileNetV2.** Once the head can roughly classify, we unfreeze 🔥 the last few backbone layers and train everything together, but with a *much* smaller learning rate. This lets the high-level features specialize toward hand gestures without erasing what MobileNetV2 already knew.

The questions below test the key design decisions. Try answering first — click **🔽 Show answer** to expand the detailed explanation.

---

**❓ Q1.  Why MobileNetV2 specifically — and not (say) ResNet-152 or training a CNN from scratch?**

- A) It's the highest-accuracy ImageNet model available.
- B) It's lightweight (~3.5M weights) and already comes with strong ImageNet-pretrained features.
- C) It's the only model PyTorch supports.
- D) It removes the need for any labelled data.

<details>
<summary>🔽 Show answer</summary>

✅ **B.** MobileNetV2 is small (~3.5M weights, vs ~60M for ResNet-152) and fast enough to run in real time. Just as importantly, it was pretrained on ImageNet, so it already understands the visual building blocks (edges, textures, common shapes) we'd otherwise need millions of images to learn. ResNet-152 (A) is far bigger than our dataset can support and would overfit immediately. Training from scratch (implied by A/D) is hopeless here: a few hundred to a few thousand gesture images is nowhere near enough to learn basic visual features from zero.

</details>

---

**❓ Q2.  In Phase 1 we *freeze* MobileNetV2's backbone and only train the new classifier head. Why?**

- A) Frozen layers train faster than unfrozen ones.
- B) The fresh head's random weights produce big, noisy gradients that would corrupt MobileNetV2's pretrained weights.
- C) PyTorch can't optimize multiple layers simultaneously.
- D) Freezing saves disk space when saving the checkpoint.

<details>
<summary>🔽 Show answer</summary>

✅ **B.** The classifier head starts as random numbers, so the early gradients (the "update signal" the optimizer uses) are huge and noisy. If we let those flow back into MobileNetV2 before the head had learned anything sensible, they'd overwrite the carefully-tuned ImageNet features. Freezing the backbone in Phase 1 means only the head adapts. Once the head is reasonable, Phase 2 unfreezes the top blocks at a much lower learning rate so the high-level features can specialize *without* being destroyed.

</details>

---

#### 🎨 Training-time image transforms

These steps run on every image *before* it reaches the model. Augmentations (the random ones) only apply during training, never during validation — we want the validation score to be a stable measurement, not a moving target.

| Transform | What it does | Why we use it |
|---|---|---|
| 📏 `Resize(224, 224)` | Rescales every image to 224×224 pixels | MobileNetV2's native input resolution — mismatched sizes would break the convolutional stack |
| 🎚️ `Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])` | For each colour channel: subtract the mean, divide by the standard deviation | These are ImageNet's statistics. The pretrained weights expect inputs in this range; matching it keeps the early features behaving as they did during pretraining |
| 🔁 `RandomHorizontalFlip` | 50 % chance to mirror left ↔ right | Cheap augmentation that effectively doubles the dataset; safe because two-hand gestures are mirror-symmetric |
| 🔄 `RandomAffine(degrees=CNN_AUG_ROT_DEG, translate=CNN_AUG_TRANSLATE)` | Small random rotation + shift | Simulates the user not being perfectly centered. Kept *small* — large rotations could change what a gesture means |
| 💡 `ColorJitter(brightness=CNN_AUG_BRIGHTNESS)` | Random brightness shift | Mild lighting variation so the model doesn't latch onto your room's exact lighting |
| 🧮 `ToTensor` | uint8 pixels in `[0..255]` → float32 in `[0..1]`, image axis order `HWC` → `CHW` | PyTorch's standard image-tensor layout |

The **validation** pipeline keeps only `Resize`, `ToTensor`, and `Normalize` — no random augmentation. 🎯

---

**❓ Q3.  Why does the head squeeze 1280-dim features into a 128-dim bottleneck before producing class logits, instead of going `Linear(1280 → num_classes)` directly?**

- A) The model can only process inputs of size ≤ 128.
- B) The bottleneck + ReLU adds non-linear capacity, and the two `Dropout` layers around it prevent overfitting on a small dataset.
- C) MobileNetV2 outputs 128 dims, not 1280.
- D) It's a hard requirement of `nn.Sequential`.

<details>
<summary>🔽 Show answer</summary>

✅ **B.** A bare `Linear(1280 → num_classes)` is just a straight-line decision rule — it can't carve out curved boundaries. Inserting `Linear → ReLU → Linear` makes the head non-linear (`ReLU` is `max(0, x)`, a simple curve), so it can learn more complex patterns. The two `Dropout` layers around the bottleneck randomly zero out activations during training — this stops the head from leaning too hard on any single feature and is one of the cheapest, most effective regularization tricks.

</details>

---

**❓ Q4.  Phase 1 uses `lr=1e-3`; Phase 2 drops to `lr=1e-5` (100× smaller). Why?**

- A) Phase 2 has fewer epochs, so each step must be smaller.
- B) The backbone holds pretrained weights worth preserving — large updates would erase them.
- C) `Adam` requires `lr=1e-5` whenever fine-tuning.
- D) GPU memory is tighter in Phase 2.

<details>
<summary>🔽 Show answer</summary>

✅ **B.** In Phase 2 we let gradients flow back into the *pretrained* MobileNetV2 blocks. Those weights are already near a good solution — we want to *nudge* them toward our task, not retrain them. A small learning rate (`1e-5`) makes tiny adjustments so the high-level features specialize from "objects in general" toward "hand gestures" without destroying the low-level features that took ImageNet weeks of compute to learn.

</details>

---

**📉 Loss & 🧭 optimizer.** `CrossEntropyLoss` is the standard loss for multi-class classification — it takes the model's raw scores, runs them through softmax to get probabilities, and penalizes the model when it's confident *and* wrong. `Adam` is an optimizer that automatically adjusts the step size per weight, so it converges quickly with little hand-tuning.

**✅ Validation.** An 80/20 random split is recomputed each epoch. We track validation accuracy and keep the **best** 🏆 checkpoint — so even if a late epoch makes things worse, we still save the best version.

#### 🏗️ 3A.1 · CNN architecture · `build_model()`


In [ ]:
# --- 3A.1  CNN architecture (single source of truth) ----------------------
# This is the ONLY place the CNN architecture is defined.  Section 3A uses
# it for training; Section 5 captures its source via inspect.getsource() and
# bundles it into the team's exported model_arch.py.  If you change the
# structure here — add a layer, swap an activation, replace the backbone —
# the change automatically flows to the exported file with no manual sync.
#
# Hyperparameters are passed as explicit arguments (not pulled from notebook
# globals) so the exported file can hard-code the literal values it was
# trained with, without re-referencing notebook-only variable names.

def build_model(num_classes: int,
                hidden_dim: int,
                dropout_1: float,
                dropout_2: float,
                pretrained: bool = False,
                freeze_base: bool = False):
    """MobileNetV2 backbone + small custom classifier head.

    Args:
        num_classes:  number of output classes.
        hidden_dim:   bottleneck width between MobileNet features and logits.
        dropout_1:    dropout after MobileNet pooling.
        dropout_2:    dropout before final logits.
        pretrained:   load ImageNet weights into the backbone (training only).
        freeze_base:  freeze MobileNetV2's parameters (Phase 1 training).
    """
    import torch.nn as nn
    from torchvision import models

    if pretrained:
        try:
            weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
            m = models.mobilenet_v2(weights=weights)
        except (AttributeError, TypeError):
            m = models.mobilenet_v2(pretrained=True)
    else:
        m = models.mobilenet_v2()

    if freeze_base:
        for p in m.parameters():
            p.requires_grad = False

    m.classifier = nn.Sequential(
        nn.Dropout(dropout_1),
        nn.Linear(1280, hidden_dim),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout_2),
        nn.Linear(hidden_dim, num_classes),
    )
    return m

print('build_model() defined.')

In [ ]:
# --- 3A.  Train CNN (MobileNetV2 fine-tune) -------------------------------
if MODEL_TYPE != 'cnn':
    print(f'Skipping CNN training (MODEL_TYPE={MODEL_TYPE}).  Section 3B will run instead.')
else:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, Subset
    from torchvision import datasets, transforms

    # ImageNet normalization constants (see the markdown table above for why).
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

    if not IMAGES_DIR.exists():
        raise FileNotFoundError(f'No images at {IMAGES_DIR}.  Run Section 2 first.')

    # ----- transforms -----------------------------------------------------
    # The training pipeline applies random augmentations (flip / rotate /
    # brightness) so the model sees slightly varied inputs each epoch.
    # The validation pipeline applies ONLY the deterministic steps — we
    # want a stable yardstick, not a moving target.
    train_tf = transforms.Compose([
        transforms.Resize(CNN_IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=CNN_AUG_ROT_DEG, translate=CNN_AUG_TRANSLATE),
        transforms.ColorJitter(brightness=CNN_AUG_BRIGHTNESS),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(CNN_IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    # ImageFolder reads each subfolder of IMAGES_DIR as a class.
    # We make TWO copies so train and val can apply different transforms.
    train_full = datasets.ImageFolder(str(IMAGES_DIR), transform=train_tf)
    val_full   = datasets.ImageFolder(str(IMAGES_DIR), transform=val_tf)
    if len(train_full) == 0 or len(train_full.classes) < 2:
        raise RuntimeError('Need at least 2 classes with images.')

    # 80/20 random split.  Using a fixed seed makes runs reproducible.
    class_names = train_full.classes
    n        = len(train_full)
    val_size = max(1, int(CNN_VAL_FRACTION * n))
    gen      = torch.Generator().manual_seed(CNN_RANDOM_SEED)
    perm     = torch.randperm(n, generator=gen).tolist()
    val_idx, train_idx = perm[:val_size], perm[val_size:]

    # DataLoader hands the model batches of CNN_BATCH_SIZE images at a time.
    # shuffle=True for training so the model sees a fresh order each epoch.
    train_loader = DataLoader(Subset(train_full, train_idx),
                              batch_size=CNN_BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(Subset(val_full,   val_idx),
                              batch_size=CNN_BATCH_SIZE, shuffle=False, num_workers=2)
    print(f'Classes: {class_names}\nTrain: {len(train_idx)}  Val: {len(val_idx)}')

    # ----- model: built via the shared helper from Section 3A -------------
    # Phase 1 freezes the MobileNetV2 backbone (freeze_base=True) so only the
    # fresh classifier head trains.  pretrained=True loads ImageNet weights.
    model = build_model(
        num_classes=len(class_names),
        hidden_dim=CNN_HIDDEN_DIM,
        dropout_1=CNN_DROPOUT_1,
        dropout_2=CNN_DROPOUT_2,
        pretrained=True,
        freeze_base=True,
    )
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()       # loss for multi-class classification

    # The canonical PyTorch training loop.  For each batch we:
    #   1. forward pass  : model(x) produces raw scores ("logits")
    #   2. compute loss  : how wrong we are vs the true label y
    #   3. backward pass : loss.backward() fills in each weight's gradient
    #   4. update step   : optim.step() nudges weights to reduce the loss
    # zero_grad() resets the gradient accumulator first (gradients otherwise
    # accumulate across batches, which we don't want here).
    # During validation we skip steps 3 and 4 and wrap in torch.no_grad()
    # so PyTorch doesn't waste memory tracking gradients we won't use.
    def _run_epoch(loader, optim, train: bool):
        model.train(train)                  # toggles dropout/batchnorm into train/eval mode
        loss_sum = correct = total = 0
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)            # forward pass
                loss   = criterion(logits, y)
                if train:
                    optim.zero_grad()        # clear old gradients
                    loss.backward()          # compute new gradients
                    optim.step()             # apply the update
                loss_sum += loss.item() * x.size(0)
                correct  += (logits.argmax(1) == y).sum().item()
                total    += y.size(0)
        return loss_sum/total, correct/total

    # ----- Phase 1: train the head only -----------------------------------
    # Only the new classifier head has requires_grad=True, so the optimizer
    # only updates the head's weights — the backbone stays frozen.
    best_acc, best_state = 0.0, None
    optim = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=CNN_LR_HEAD)
    print(f'\n[Phase 1] Train head — {CNN_EPOCHS} epochs (lr={CNN_LR_HEAD})')
    for ep in range(1, CNN_EPOCHS + 1):
        tl, ta = _run_epoch(train_loader, optim, True)
        vl, va = _run_epoch(val_loader,   optim, False)
        print(f'  ep {ep:>2}/{CNN_EPOCHS}  train_loss={tl:.4f} acc={ta:.1%}  |  val_loss={vl:.4f} acc={va:.1%}')
        # Keep the best checkpoint so noisy late epochs can't make things worse.
        if va > best_acc:
            best_acc, best_state = va, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)

    # ----- Phase 2: unfreeze the top blocks and fine-tune ------------------
    # Now we let gradients flow into the top of MobileNetV2 — but at a 100x
    # smaller learning rate so we only nudge the pretrained weights gently.
    if CNN_FINETUNE_EPOCHS > 0:
        for layer in list(model.features.children())[-CNN_UNFREEZE_LAST_N:]:
            for p in layer.parameters():
                p.requires_grad = True
        optim = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=CNN_LR_FINETUNE)
        print(f'\n[Phase 2] Fine-tune top {CNN_UNFREEZE_LAST_N} blocks — '
              f'{CNN_FINETUNE_EPOCHS} epochs (lr={CNN_LR_FINETUNE})')
        for ep in range(1, CNN_FINETUNE_EPOCHS + 1):
            tl, ta = _run_epoch(train_loader, optim, True)
            vl, va = _run_epoch(val_loader,   optim, False)
            print(f'  ep {ep:>2}/{CNN_FINETUNE_EPOCHS}  train_loss={tl:.4f} acc={ta:.1%}  |  val_loss={vl:.4f} acc={va:.1%}')
            if va > best_acc:
                best_acc, best_state = va, {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if best_state is not None:
            model.load_state_dict(best_state)

    # ----- save the best checkpoint in the format expected by the game ----
    out_path = MODELS_OUT_DIR / 'hand_sign_cnn.pth'
    torch.save({
        'state_dict':  {k: v.cpu() for k, v in model.state_dict().items()},
        'class_names': class_names,
        'img_size':    list(CNN_IMG_SIZE),
    }, str(out_path))
    print(f'\nBest val accuracy: {best_acc:.1%}\nSaved -> {out_path}')

### 🏋️ 3B · Landmark training (RandomForest / MLP) ✋

#### 💬 What's happening in plain English

Instead of feeding raw images to a deep neural network, we let **MediaPipe** 📐 do the hard "where is the hand, where are its joints?" work first. It returns 21 keypoints per hand (wrist, knuckles, fingertips, …), and after we normalize them we're left with just **126 numbers per gesture** — small enough that a simple, classical classifier handles the patterns just fine:

- 🌲 **RandomForest (default):** a committee of decision trees that each look at the data slightly differently and vote on the answer. Robust, fast, no GPU needed.
- 🧠 **MLP (alternative):** a small neural network. Sometimes a bit more accurate if you have lots of data.

Either way, training is done in seconds. ⚡ The questions below test why this simpler approach works. Try answering first — click **🔽 Show answer** to expand.

---

**❓ Q1.  Why can a classical sklearn classifier work here, when the CNN pipeline needs a deep neural network?**

- A) MediaPipe already solved the hard "where is the hand?" problem — only a simple geometric classification remains.
- B) RandomForest is strictly more accurate than any neural network.
- C) Landmark coordinates can't be processed by neural networks.
- D) sklearn is faster than PyTorch on GPU.

<details>
<summary>🔽 Show answer</summary>

✅ **A.** MediaPipe's HandLandmarker already does the heavy lifting — finding each hand in the image and producing 21 keypoints with (x, y, z) coordinates. What's left is the much easier problem "given 126 normalized numbers, which gesture is this?" That's a small, low-dimensional pattern-recognition problem — well within classical machine learning's comfort zone, with way less data needed and no GPU required. It's also why no image augmentation is necessary: the normalization step (below) already removes most of the variation augmentation would fight.

</details>

---

#### 🎯 Feature normalization recap (done in Section 2B.3)

| Step | Operation | Property gained |
|---|---|---|
| 1️⃣ | Subtract the wrist (landmark 0) from every point | **Translation-invariant** — the same gesture at any position on screen produces the same features |
| 2️⃣ | Divide every coordinate by the wrist → middle-finger-MCP distance (landmark 9) | **Scale-invariant** — the same gesture from different distances or by different hand sizes produces the same features |
| 3️⃣ | Sort the two hands by their leftmost x-coordinate | **Slot-consistent** — the left hand's features always land in the "left" slot, no matter what order MediaPipe returned them in |
| 4️⃣ | Concatenate hand-1 and hand-2 vectors | Final: 21 × 3 × 2 = **126 numbers per sample** |

Because inputs are already invariant to position / size / distance, **no augmentation is needed** — the classifier just needs to learn the geometric signature of each gesture. ✨

---

**❓ Q2.  Why is RandomForest a reasonable default over an MLP / deep neural network for this problem?**

- A) It's the only sklearn classifier that handles 5-class problems.
- B) It's robust to noise, needs no feature scaling, outputs probabilities, and trains in seconds.
- C) It's strictly more accurate than every neural network on every problem.
- D) Neural networks can't output probabilities.

<details>
<summary>🔽 Show answer</summary>

✅ **B.** A RandomForest is a *committee of decision trees*. Each tree is trained on a slightly different slice of the data and looks at a random subset of features at each branch — so the trees disagree slightly, and averaging their votes cancels out individual mistakes. You don't need to scale the features, you don't need to tune much (`n_estimators` and `max_depth` cover most cases), and `predict_proba()` gives a probability for each class — which `inference.py` uses to ignore low-confidence predictions in the game. The MLP option exists too — with ≥ 1000 samples per class it can edge out the forest — but the forest is the safer baseline.

</details>

---

**❓ Q3.  Why does `train_test_split(..., stratify=y)` matter here?**

- A) It speeds up the split.
- B) It guarantees identical class proportions in train and test, so the reported accuracy is meaningful even with imbalanced classes.
- C) It encrypts the test labels.
- D) It's required by sklearn.

<details>
<summary>🔽 Show answer</summary>

✅ **B.** A plain random split could, by bad luck, put almost all of `move5` in the test set — making the reported "accuracy" mostly about how well we predict `move5`. Stratifying on the label keeps the same class proportions in *both* halves, so 90 % test accuracy really does mean ~90 % across every class, not skewed by an over-represented one.

</details>

---

**🧠 Why MLP (alternative).** A small fully-connected neural network (`MLP_HIDDEN → … → classes`). It can learn smoother decision boundaries than RandomForest and tends to generalize a little better with lots of data. `early_stopping=True` watches a held-out validation slice and halts training when the validation score stops improving — automatic overfitting prevention.

**💾 Outputs (drop-in for the local game):**

- 📦 `hand_sign_classifier.pkl` — the fitted classifier
- 🏷️ `label_encoder.pkl` — translates numeric class indices ↔ the `'move1' … 'move5'` strings

In [ ]:
# --- 3B.  Train landmark classifier (RandomForest / MLP) -----------------
if MODEL_TYPE != 'landmark':
    print(f'Skipping landmark training (MODEL_TYPE={MODEL_TYPE}).  Section 3A will run instead.')
else:
    import csv, joblib, numpy as np
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neural_network import MLPClassifier
    from sklearn.metrics import accuracy_score, classification_report
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder

    if not LANDMARK_CSV.exists():
        raise FileNotFoundError(f'No CSV at {LANDMARK_CSV}.  Run Section 2 first.')

    # ----- load CSV: each row = [class_label, f0, f1, ..., f125] ---------
    labels, feats = [], []
    with open(LANDMARK_CSV) as f:
        for row in csv.reader(f):
            if not row: continue
            labels.append(row[0])
            feats.append([float(x) for x in row[1:]])
    X = np.array(feats); y_raw = np.array(labels)
    print(f'{len(X)} samples, {len(set(y_raw))} classes, {X.shape[1]} features')

    # Encode 'move1'..'move5' as 0..4 so sklearn can learn from them.
    le = LabelEncoder(); y = le.fit_transform(y_raw)
    print(f'Classes: {list(le.classes_)}')

    # Stratified split preserves class balance in both halves.
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=LANDMARK_TEST_SIZE, random_state=LANDMARK_SEED, stratify=y)
    print(f'Train: {len(Xtr)}   Test: {len(Xte)}')

    # ----- pick the classifier -------------------------------------------
    if LANDMARK_MODEL == 'rf':
        clf = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            random_state=LANDMARK_SEED, n_jobs=-1)
    elif LANDMARK_MODEL == 'mlp':
        clf = MLPClassifier(
            hidden_layer_sizes=MLP_HIDDEN, activation='relu',
            max_iter=MLP_MAX_ITER, early_stopping=MLP_EARLY_STOP,
            random_state=LANDMARK_SEED)
    else:
        raise ValueError(f'Unknown LANDMARK_MODEL: {LANDMARK_MODEL}')

    print(f'\nTraining {type(clf).__name__} ...')
    clf.fit(Xtr, ytr)
    yp = clf.predict(Xte)
    print(f'\nAccuracy: {accuracy_score(yte, yp):.1%}\n')
    print(classification_report(yte, yp, target_names=le.classes_))

    model_path   = MODELS_OUT_DIR / 'hand_sign_classifier.pkl'
    encoder_path = MODELS_OUT_DIR / 'label_encoder.pkl'
    joblib.dump(clf, model_path)
    joblib.dump(le,  encoder_path)
    print(f'\nSaved -> {model_path}\nSaved -> {encoder_path}')

---
## 🔮 Section 4 · Inference (sanity check)

Quick prediction to confirm the trained model loads and classifies correctly. UDP send to the local game socket is intentionally skipped — you do that on your PC with the original `inference.py` after dropping the downloaded weights into `Teams/Team<N>/models/`. 🎮

The model-specific `predict()` function is built in **4A** (CNN) or **4B** (landmark). The test cell **4D** (webcam snapshot) then calls whichever `predict()` got built.

### 🖼️ 4A · Build CNN predictor

Reload the saved `.pth`, rebuild the exact MobileNetV2 + custom-head architecture (you have to recreate the structure before `load_state_dict()` will accept the saved weights), then expose `predict(frame_bgr)` that returns `(class_name, confidence)`. We apply the **same** preprocessing the model saw during training — same resize, same ImageNet normalization — otherwise predictions would degrade silently. ⚠️

In [ ]:
# --- 4A.  Build CNN predict() --------------------------------------------
if MODEL_TYPE != 'cnn':
    print(f'Skipping (MODEL_TYPE={MODEL_TYPE}).')
else:
    import torch, numpy as np, cv2
    from torchvision import models
    import torch.nn as nn, torch.nn.functional as F

    ckpt = torch.load(str(MODELS_OUT_DIR / 'hand_sign_cnn.pth'),
                      map_location=DEVICE, weights_only=False)
    CNAMES   = ckpt['class_names']
    IMG_SIZE = tuple(ckpt.get('img_size', (224, 224)))
    _MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    _STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    # Recreate the exact training architecture before loading weights.
    _m = models.mobilenet_v2()
    _m.classifier = nn.Sequential(
        nn.Dropout(CNN_DROPOUT_1), nn.Linear(1280, CNN_HIDDEN_DIM), nn.ReLU(inplace=True),
        nn.Dropout(CNN_DROPOUT_2), nn.Linear(CNN_HIDDEN_DIM, len(CNAMES)))
    _m.load_state_dict(ckpt['state_dict']); _m.to(DEVICE).eval()

    def predict(frame_bgr):
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        img = cv2.resize(rgb, IMG_SIZE).astype(np.float32) / 255.0
        img = (img - _MEAN) / _STD
        t   = torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            p = F.softmax(_m(t), dim=1).cpu().numpy()[0]
        i = int(np.argmax(p))
        return CNAMES[i], float(p[i])
    print('CNN predict() ready.')

### ✋ 4B · Build landmark predictor

Reload the sklearn classifier + label encoder, rebuild the MediaPipe detector, and expose the same `predict(frame_bgr)` interface. Returns `(None, 0.0)` when two hands aren't visible — the local `inference.py` uses this to require a clean two-hand pose before sending an action to the game. 🖐️🤚

In [ ]:
# --- 4B.  Build landmark predict() ---------------------------------------
if MODEL_TYPE != 'landmark':
    print(f'Skipping (MODEL_TYPE={MODEL_TYPE}).')
else:
    import joblib, numpy as np, cv2
    import mediapipe as mp
    from mediapipe import tasks as mp_tasks

    LANDMARKER_PATH = WORK_DIR / 'hand_landmarker.task'
    if not LANDMARKER_PATH.exists():
        import urllib.request
        url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
        urllib.request.urlretrieve(url, LANDMARKER_PATH)
    _opts = mp_tasks.vision.HandLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=str(LANDMARKER_PATH)),
        num_hands=4, min_hand_detection_confidence=0.7, min_tracking_confidence=0.6)
    _DET = mp_tasks.vision.HandLandmarker.create_from_options(_opts)

    _CLF = joblib.load(MODELS_OUT_DIR / 'hand_sign_classifier.pkl')
    _LE  = joblib.load(MODELS_OUT_DIR / 'label_encoder.pkl')

    def _bbox(lms):
        xs=[l.x for l in lms]; ys=[l.y for l in lms]
        return (max(xs)-min(xs))*(max(ys)-min(ys))
    def _two(res):
        if not res.hand_landmarks or len(res.hand_landmarks)<2: return None
        t = sorted(res.hand_landmarks, key=_bbox, reverse=True)[:2]
        t.sort(key=lambda l: l[0].x); return t
    def _norm(lms):
        a=np.array([[l.x,l.y,l.z] for l in lms]); c=a-a[0]
        s=np.linalg.norm(c[9])
        if s>1e-6: c/=s
        return c.flatten()

    def predict(frame_bgr):
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        hands = _two(_DET.detect(img))
        if hands is None:
            return None, 0.0          # not enough hands -> abstain
        feats = np.concatenate([_norm(hands[0]), _norm(hands[1])]).reshape(1,-1)
        probs = _CLF.predict_proba(feats)[0]
        i = int(np.argmax(probs))
        return _LE.classes_[i], float(probs[i])
    print('Landmark predict() ready.')

### 📷 4C · Predict from a webcam snapshot


In [ ]:
# --- 4C.  Predict from a webcam snapshot ---------------------------------
# Requires the webcam helpers from Section 2B.1.
import time
webcam_start()
print('Hold gesture, capturing in 2 s ...'); time.sleep(2)
frame = webcam_grab_bgr()
webcam_stop()
label, conf = predict(frame)
if label is None:
    print('(no two-hand pose detected)')
else:
    print(f'Prediction: {label}  ({conf:.0%})')


---
## 💾 Section 5 · Download the trained weights (and architecture)

Packs everything the local game needs into one zip and triggers a browser download.

For **🖼️ CNN**, the bundle contains *two* things:

| File | Where it goes after unzip | What it is |
|---|---|---|
| 🏗️ `model_arch.py` | `Teams/Team<N>/model_arch.py` | A small Python file describing your exact architecture, **auto-captured from Section 3A's `build_model()` via `inspect.getsource()`**. Whatever you change in Section 3A (hyperparameters, layers, backbone, activations) flows verbatim into this file — no manual edits required. The local `inference.py`, `train_model.py`, and `game.py` import this and call its `build_model()` to rebuild the model before loading the weights. Without it, they fall back to the hard-coded default architecture, which only works if Section 3A still matches the default. |
| 🎛️ `models/hand_sign_cnn.pth` | `Teams/Team<N>/models/hand_sign_cnn.pth` | The trained weights (state_dict + class_names + img_size). |

For **✋ Landmark**, only the weights need to ship — sklearn classifiers serialize with their own structure, so no architecture file is needed:

| File | Where it goes after unzip |
|---|---|
| 📦 `models/hand_sign_classifier.pkl` | `Teams/Team<N>/models/hand_sign_classifier.pkl` |
| 🏷️ `models/label_encoder.pkl` | `Teams/Team<N>/models/label_encoder.pkl` |

> 💡 **Why the architecture file?** The CNN's `state_dict` is just a bag of weight tensors with shapes baked in. The local code has to know the architecture *before* it can load those weights. Shipping `model_arch.py` lets each team customize their architecture without touching the game's code — and because Section 5 captures Section 3A's `build_model()` source directly, the exported file can never drift from what was actually trained.

In [ ]:
# --- 5.  Write model_arch.py (CNN only), then zip + download --------------
import inspect, shutil
from google.colab import files

team_root = MODELS_OUT_DIR.parent          # = WORK_DIR/Teams/Team{N}/

# 5.1 — For CNN, generate a model_arch.py that bundles the team's exact
# architecture so code/game.py + code/model/cnn/{inference,train_model}.py
# can rebuild it before loading the weights.  Skipped for landmark (sklearn
# pickles are self-describing).
#
# Single source of truth: we capture Section 3A's `build_model()` via
# inspect.getsource() and rename it to _build_arch_impl in the exported
# file, then add a thin wrapper named build_model that hard-codes the
# hyperparameters this team trained with.  Any structural change to
# Section 3A automatically flows into the exported file with no manual
# edits.

if MODEL_TYPE == 'cnn':
    captured = inspect.getsource(build_model)
    # Rename the captured function so the file can expose its own
    # `build_model(num_classes, ...)` wrapper that game.py looks for.
    captured = captured.replace('def build_model(', 'def _build_arch_impl(', 1)

    header = f'''"""
Team {TEAM_NUM} - custom CNN architecture
Generated by sigil_strike_colab.ipynb.

The local game (code/game.py) imports this file from
Teams/Team{TEAM_NUM}/model_arch.py when present, and calls build_model()
to reconstruct the architecture before loading the .pth weights.

The architecture body below was captured verbatim from the notebook's
Section 3A's `build_model()` via inspect.getsource() at training time - there
is no separate copy to keep in sync.  The hyperparameter literals below
were the values used to train models/hand_sign_cnn.pth; do not edit them
unless you also retrain (load_state_dict will fail on shape mismatch).
"""
from __future__ import annotations

# Hyperparameters captured at training time.
HIDDEN_DIM = {CNN_HIDDEN_DIM}
DROPOUT_1  = {CNN_DROPOUT_1}
DROPOUT_2  = {CNN_DROPOUT_2}


'''

    wrapper = '''

def build_model(num_classes: int,
                pretrained: bool = False,
                freeze_base: bool = False):
    """Game-facing entry point: rebuilds the architecture with the
    hyperparameters this team trained with."""
    return _build_arch_impl(
        num_classes=num_classes,
        hidden_dim=HIDDEN_DIM,
        dropout_1=DROPOUT_1,
        dropout_2=DROPOUT_2,
        pretrained=pretrained,
        freeze_base=freeze_base,
    )
'''

    arch_path = team_root / 'model_arch.py'
    arch_path.write_text(header + captured + wrapper, encoding='utf-8')
    print(f'Wrote {arch_path}')

# 5.2 — Sanity check: do we actually have weights to ship?
produced = sorted(MODELS_OUT_DIR.glob('*'))
if not produced:
    raise FileNotFoundError(f'No weights found in {MODELS_OUT_DIR}.  Run Section 3 first.')

# 5.3 — Show the user what's about to ship
print('\nBundle contents:')
for p in sorted(team_root.rglob('*')):
    if p.is_file():
        rel = p.relative_to(team_root)
        print(f'  {rel}  ({p.stat().st_size/1024:.1f} KB)')

# 5.4 — Zip from the TEAM root so the unzip layout matches Teams/Team{N}/
zip_base = WORK_DIR / f'Team{TEAM_NUM}_{MODEL_TYPE}_bundle'
zip_path = pathlib.Path(shutil.make_archive(str(zip_base), 'zip', root_dir=team_root))
print(f'\nZipped -> {zip_path.name}')
print(f'On your PC: unzip into  Teams/Team{TEAM_NUM}/  (overwrite when prompted)')
print('Starting download ...')
files.download(str(zip_path))